# Train InstructPix2Pix from Stable Diffusion 1.5

Run cells in order. MagicBrush's train split is streamed so the entire dataset is not downloaded first. Training is best on a CUDA GPU; lower `resolution` if memory is limited.


In [86]:
import os
from pathlib import Path

import torch
import torch.nn.functional as F
from datasets import load_dataset
from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel
from dotenv import load_dotenv
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm.auto import tqdm
from transformers import CLIPTextModel, CLIPTokenizer

load_dotenv()
torch.manual_seed(0)


In [87]:
model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"
resolution = 256  # Divisible by 8; 512 uses much more memory.
batch_size = 1
gradient_accumulation_steps = 4
max_train_steps = 500
learning_rate = 5e-5
conditioning_dropout_prob = 0.05
max_grad_norm = 1.0
save_every = 100
output_dir = Path("checkpoints/diffusers-ip2p")

device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else "cpu"
)
print("Training device:", device)


Training device: mps


## Data

Each record is an original image, an edited image, and an instruction. Both images receive the same deterministic resize and normalization; instructions are tokenized for CLIP. The mask and metadata are left out of the batch.


In [88]:
train_dataset = load_dataset(
    "osunlp/MagicBrush",
    data_files={"train": "data/train-*"},
    split="train",
    streaming=True,
    token=os.getenv("HF_TOKEN"),
)

image_transform = transforms.Compose([
    transforms.Resize((resolution, resolution), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),  # PIL RGB -> float tensor in [0, 1]
    transforms.Normalize([0.5] * 3, [0.5] * 3),  # -> [-1, 1]
])

def collate_fn(rows):
    return {
        "original_pixel_values": torch.stack([
            image_transform(row["source_img"].convert("RGB")) for row in rows
        ]),
        "edited_pixel_values": torch.stack([
            image_transform(row["target_img"].convert("RGB")) for row in rows
        ]),
        "input_ids": tokenizer(
            [row["instruction"] for row in rows],
            padding="max_length",
            truncation=True,
            max_length=tokenizer.model_max_length,
            return_tensors="pt",
        ).input_ids,
    }


## Model

InstructPix2Pix feeds the UNet four noisy target latent channels and four original image latent channels. Copy Stable Diffusion's existing convolution weights and initialize the new channels to zero. Only the UNet is trainable.


In [89]:
tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(model_id, subfolder="text_encoder").to(device)
vae = AutoencoderKL.from_pretrained(model_id, subfolder="vae").to(device)
unet = UNet2DConditionModel.from_pretrained(model_id, subfolder="unet")
noise_scheduler = DDPMScheduler.from_pretrained(model_id, subfolder="scheduler")
assert noise_scheduler.config.prediction_type == "epsilon"

old_conv = unet.conv_in
new_conv = nn.Conv2d(
    8, old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding,
)
with torch.no_grad():
    new_conv.weight.zero_()
    new_conv.weight[:, :4].copy_(old_conv.weight)
    new_conv.bias.copy_(old_conv.bias)
unet.conv_in = new_conv
unet.register_to_config(in_channels=8)  # Keep saved UNet config in sync.

vae.requires_grad_(False).eval()
text_encoder.requires_grad_(False).eval()
unet.to(device).train()
unet.enable_gradient_checkpointing()


Loading weights: 100%|██████████| 196/196 [00:00<00:00, 5835.72it/s]
/Users/xyz/opt/miniconda3/envs/image-edit/lib/python3.14/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


In [90]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, collate_fn=collate_fn)

# Check a real batch before the first training step.
batch = next(iter(train_loader))
assert batch["original_pixel_values"].shape == (batch_size, 3, resolution, resolution)
assert batch["edited_pixel_values"].shape == (batch_size, 3, resolution, resolution)
assert batch["input_ids"].shape == (batch_size, tokenizer.model_max_length)
for key in ("original_pixel_values", "edited_pixel_values"):
    assert batch[key].min() >= -1 and batch[key].max() <= 1
print({key: tuple(value.shape) for key, value in batch.items()})


{'original_pixel_values': (1, 3, 256, 256), 'edited_pixel_values': (1, 3, 256, 256), 'input_ids': (1, 77)}


## Diffusion loss

The edited target uses a sampled VAE posterior multiplied by the VAE scaling factor. The source condition uses the posterior mode without that factor. Drop text, image, or both conditions on a fraction of examples so the model can use dual classifier-free guidance at inference.


In [91]:
@torch.no_grad()
def encode_target(batch):
    pixels = batch["edited_pixel_values"].to(device)
    return vae.encode(pixels).latent_dist.sample() * vae.config.scaling_factor

@torch.no_grad()
def encode_conditions(batch):
    text_hidden = text_encoder(batch["input_ids"].to(device))[0]
    source_pixels = batch["original_pixel_values"].to(device)
    source_latents = vae.encode(source_pixels).latent_dist.mode()
    return text_hidden, source_latents

@torch.no_grad()
def encode_empty_prompt():
    empty_ids = tokenizer(
        [""], padding="max_length", truncation=True,
        max_length=tokenizer.model_max_length, return_tensors="pt",
    ).input_ids.to(device)
    return text_encoder(empty_ids)[0]

null_text_hidden = encode_empty_prompt()

def training_step(batch):
    target_latents = encode_target(batch)
    noise = torch.randn_like(target_latents)
    timesteps = torch.randint(
        0, noise_scheduler.config.num_train_timesteps,
        (target_latents.shape[0],), device=device,
    ).long()
    noisy_latents = noise_scheduler.add_noise(target_latents, noise, timesteps)
    text_hidden, source_latents = encode_conditions(batch)

    r = torch.rand(target_latents.shape[0], device=device)
    drop_text = r < 2 * conditioning_dropout_prob
    drop_source = (r >= conditioning_dropout_prob) & (r < 3 * conditioning_dropout_prob)
    text_hidden = torch.where(
        drop_text[:, None, None], null_text_hidden.expand_as(text_hidden), text_hidden
    )
    source_latents = source_latents * (~drop_source)[:, None, None, None]

    model_input = torch.cat([noisy_latents, source_latents], dim=1)
    prediction = unet(model_input, timesteps, encoder_hidden_states=text_hidden).sample
    return F.mse_loss(prediction.float(), noise.float())


## Optimize and save

`max_train_steps` counts optimizer updates. The effective batch size is `batch_size * gradient_accumulation_steps`. Checkpoints save the 8-channel UNet; use the same Stable Diffusion 1.5 components when loading it.


In [92]:
optimizer = torch.optim.AdamW(unet.parameters(), lr=learning_rate)
optimizer.zero_grad(set_to_none=True)
use_cuda_bf16 = device.type == "cuda" and torch.cuda.is_bf16_supported()
step = 0
micro_step = 0
running_loss = 0.0
progress = tqdm(total=max_train_steps, desc="InstructPix2Pix training")

while step < max_train_steps:
    epoch_batches = 0
    for batch in train_loader:
        epoch_batches += 1
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=use_cuda_bf16):
            loss = training_step(batch)
        (loss / gradient_accumulation_steps).backward()
        running_loss += loss.detach().item()
        micro_step += 1
        if micro_step % gradient_accumulation_steps != 0:
            continue

        torch.nn.utils.clip_grad_norm_(unet.parameters(), max_grad_norm)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        step += 1
        progress.update(1)
        if step % 10 == 0:
            progress.set_postfix(loss=f"{running_loss / (10 * gradient_accumulation_steps):.4f}")
            running_loss = 0.0
        if step % save_every == 0:
            unet.save_pretrained(output_dir / f"step-{step}" / "unet")
        if step >= max_train_steps:
            break
    if epoch_batches == 0:
        raise RuntimeError("The training dataset yielded no batches.")

progress.close()
unet.save_pretrained(output_dir / "final" / "unet")
print("Saved:", output_dir / "final" / "unet")


InstructPix2Pix training: 100%|██████████| 500/500 [1:25:54<00:00, 10.31s/it, loss=0.0831]


Saved: checkpoints/diffusers-ip2p/final/unet
